In [ ]:
import os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
# Load the required packages
import scanpy as sc
import torch
import scarches as sca
from scarches.dataset.trvae.data_handling import remove_sparsity
import matplotlib.pyplot as plt
import numpy as np
import gdown
import anndata as ad
import time
import tracemalloc
import os
import psutil

In [ ]:
# Load data
adata = sc.read_h5ad("./merged_x_prep_FINAL.h5ad") # Specify the data (x) that will be used for training the model

In [ ]:
# Standardize some of the metadata variables
adata.obs["assay"]
adata.obs.drop('ident', axis = 1)
adata.obs["assay"] = adata.obs["assay"].astype("category")
adata.obs["new_id"] = adata.obs["new_id"].astype("category")
adata.obs["tissue"] = adata.obs["tissue"].astype("category")

In [ ]:
# Converting the expression matrix to the Compressed Sparse Row format for more efficient processing
adata.X = adata.X.tocsr()

In [13]:
# Setup for scVI in scArches
sca.models.SCVI.setup_anndata(adata, batch_key='assay',categorical_covariate_keys=["new_id"])

In [ ]:
# Train the model with the parameters as per developer's reccomendations 
model = sca.models.SCVI(
    adata,
    n_layers=2,
    encode_covariates=True,
    deeply_inject_covariates=False,
    use_layer_norm="both",
    use_batch_norm="none",
)
model.train(max_epochs=75, accelerator = "mps")

In [ ]:
# Save the model
ref_path = 'D_sca_cov_model_v3/' # Specify the Dataset as D_
model.save(ref_path, overwrite=True)

# Query Model

In [ ]:
# 1. Load the new datasets (unseen by the original model)
new_adata = sc.read_h5ad("./prep_FINAL.h5ad")

In [ ]:
# Keep only the selected metadata columns
keep = ["nCount_RNA", "nFeature_RNA", "new_id", "cell_type_ontology_term_id",
        "assay", "disease", "tissue"]
new_adata.obs = new_adata.obs[keep]

# Modify the assay column 
new_adata.obs["assay"] = new_adata.obs["assay"].apply(lambda x: "3X" if x == "10x 3' v3" else "5X")
new_adata.obs["new_id"] = new_adata.obs["new_id"].astype("category")
new_adata.obs["assay"] = new_adata.obs["assay"].astype("category")


In [8]:
# --- 1.1. Split the anndata object into a disctionary of objects based on the individual id
split_column = 'new_id'
categories = new_adata.obs[split_column].unique()

# Create a dictionary to hold the split AnnData objects
split_adata_dict = {
    category: new_adata[new_adata.obs[split_column] == category, :].copy()
    for category in categories
}

print(f"Split AnnData into {len(split_adata_dict)} objects.")

Split AnnData into 9 objects.


In [ ]:
# Set up the early stopping criteria based on the developer's reccomendations 
early_stopping_kwargs = {
    "early_stopping": True,
    "early_stopping_monitor": "elbo_validation",
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.0,
    "check_val_every_n_epoch": 1,
}

In [ ]:
# Set up the memory tracker
process = psutil.Process(os.getpid())

def rss_mb():
    return process.memory_info().rss / 1024 / 1024


In [ ]:
# --- 1.2. Loop through and perform operations ---
ref_path = 'D_sca_cov_model_v3/' # Specify the Dataset with the D_

start = time.perf_counter() # track time
tracemalloc.start() # track memory
print("RSS before:", rss_mb(), "MB")

# Loop through each category and its corresponding AnnData object
for category, subset_adata in split_adata_dict.items():
    print(f"\nProcessing subset for category: '{category}'")
    
    # Align the genes in the new data to the training data
    sca.models.SCVI.prepare_query_anndata(subset_adata, ref_path)
    
    #  Map to trained model
    query_model = sca.models.SCVI.load_query_data(
    subset_adata,
    ref_path,
    freeze_dropout = True,
    )
    
    # Train the model
    query_model.train(max_epochs=400, accelerator='mps', **early_stopping_kwargs)

    # Predict the gene expression in 5X
    predicted_5 = query_model.get_normalized_expression(
    subset_adata,
    transform_batch="5X",
    library_size=10000 
    )

    end = time.perf_counter()
    print(f"Total loop runtime: {end - start:.4f} seconds")
    current, peak = tracemalloc.get_traced_memory()
    print(f"Peak memory during loop: {peak / 1024 / 1024:.2f} MB")
    print("RSS after:", rss_mb(), "MB")
    
    # Store the modified AnnData object in a new dictionary
    subset_adata.layers["predicted"] = predicted_5
    split_adata_dict[category] = subset_adata

print(f"Peak memory after loop: {peak / 1024 / 1024:.2f} MB")
tracemalloc.stop()

In [ ]:
# 2. Concatenate the modified objects back together ---
merged_adata = ad.concat(
    split_adata_dict,
    join='outer',
    label='original_batch',
    fill_value=0
)

In [ ]:
# 3. Save the merged object
merged_adata.write("./DS_scArches.h5ad")